# VELNOR-Core-Prime Fine-Tuning (Free GPU / Colab)

This notebook trains **VELNOR-Core-Prime** using **Qwen/Qwen2.5-7B-Instruct** with 4-bit NF4 QLoRA ($r=16, \alpha=32$).

It fits entirely within a free Google Colab **Tesla T4 (15 GiB)** runtime and outputs a production-ready LoRA adapter.

### 1. Verify GPU Preflight

In [ ]:
import torch
assert torch.cuda.is_available(), "No CUDA GPU detected! In Colab, go to Runtime -> Change runtime type -> T4 GPU."
p = torch.cuda.get_device_properties(0)
free, total = torch.cuda.mem_get_info(0)
print(f"GPU: {p.name} | Total: {round(total/1024**3, 2)} GiB | Free: {round(free/1024**3, 2)} GiB")

### 2. Clone Latest Velnor Repository & Install Requirements

In [ ]:
import subprocess, os, shutil

REPO_DIR = '/content/velnor'
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone --depth 1 https://github.com/vectolith/velnor.git {REPO_DIR}
%cd {REPO_DIR}

!pip install -q -r training/requirements-qwen3-qlora.txt

### 3. Optional: Mount Google Drive (to automatically persist adapters)

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/VELNOR-training-runs/final_models/VELNOR-Core-Prime'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print("Google Drive mounted. Checkpoints will mirror to:", DRIVE_DIR)
except Exception as e:
    DRIVE_DIR = None
    print("Drive not mounted. Outputs will be saved locally in /content/outputs.")

### 4. Execute VELNOR-Core-Prime Fine-Tuning

In [ ]:
import os

OUTPUT_DIR = '/content/outputs/VELNOR-Core-Prime'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Configure environment for VELNOR-Core-Prime
os.environ['VELNOR_QWEN3_BASE_MODEL'] = 'Qwen/Qwen2.5-7B-Instruct'
os.environ['VELNOR_QWEN3_OUTPUT'] = OUTPUT_DIR
os.environ['VELNOR_QWEN3_MAX_SEQ_LEN'] = '512'
os.environ['VELNOR_QWEN3_EPOCHS'] = '2'
os.environ['VELNOR_QWEN3_GRAD_ACCUM'] = '8'
os.environ['VELNOR_QWEN3_LR'] = '2e-4'
os.environ['VELNOR_QWEN3_LORA_R'] = '16'
os.environ['VELNOR_QWEN3_LORA_ALPHA'] = '32'

print("Starting VELNOR-Core-Prime training pass...")
!python -u training/train_qwen3_8b_qlora.py

### 5. Review Training & Evaluation Report

In [ ]:
import json
report_path = os.path.join(OUTPUT_DIR, 'training_report.json')
if os.path.exists(report_path):
    with open(report_path, 'r') as f:
        report = json.load(f)
    print("--- Training Summary ---")
    print("Base Model:", report.get('base_model'))
    print("Best Epoch:", report.get('validation', {}).get('best_epoch'))
    print("Best Loss:", report.get('validation', {}).get('best_loss'))
    print("Rubric Tuned Pass Rate:", report.get('rubric', {}).get('tuned', {}).get('pass_rate'))
    print("Delta vs Base:", report.get('rubric', {}).get('delta_passed'))
else:
    print("Report not generated yet.")

### 6. Export Adapter to Drive & Trigger Direct Download

In [ ]:
import shutil
from google.colab import files

adapter_source = os.path.join(OUTPUT_DIR, 'adapter')

# 1. Mirror to Drive if mounted
if DRIVE_DIR and os.path.exists(adapter_source):
    shutil.copytree(adapter_source, DRIVE_DIR, dirs_exist_ok=True)
    print(f"Copied adapter to Google Drive: {DRIVE_DIR}")

# 2. Zip and trigger browser download
zip_target = '/content/VELNOR-Core-Prime-adapter'
shutil.make_archive(zip_target, 'zip', adapter_source)
print("Downloading VELNOR-Core-Prime-adapter.zip to your local computer...")
files.download(f"{zip_target}.zip")